# ERCOT Load — Staging Layer (Raw → Cleaned)

This step:
- Removes trailing "DST" labels
- Normalizes utility-style "24:00" to next-day "00:00"
- Parses timestamps safely
- Casts all zone columns to numeric
- Preserves a DST audit flag

Output:
- `ercot.stg_ercot_load`


In [0]:
{{ config(materialized='view') }}

select
  -- keep timestamp clean
  cast(hour_ts as timestamp) as hour_ts,

  -- MW fields: strip commas + safe cast to numeric (DuckDB)
  try_cast(replace(cast(ercot_mw  as varchar), ',', '') as double) as ercot_mw,
  try_cast(replace(cast(coast_mw as varchar), ',', '') as double) as coast_mw,
  try_cast(replace(cast(east_mw  as varchar), ',', '') as double) as east_mw,
  try_cast(replace(cast(fwest_mw as varchar), ',', '') as double) as fwest_mw,
  try_cast(replace(cast(north_mw as varchar), ',', '') as double) as north_mw

  -- add the rest if your seed has them:
  -- , try_cast(replace(cast(ncent_mw as varchar), ',', '') as double) as ncent_mw
  -- , try_cast(replace(cast(south_mw as varchar), ',', '') as double) as south_mw
  -- , try_cast(replace(cast(scent_mw as varchar), ',', '') as double) as scent_mw
  -- , try_cast(replace(cast(west_mw  as varchar), ',', '') as double) as west_mw

from {{ ref('ercot_load_1h') }}
where hour_ts is not null

# Hourly Normalization Layer

Goal:
- Ensure exactly one row per hour
- Average duplicate DST hours
- Produce stable hourly time series

Output:
- `ercot.stg_ercot_load_1h`


In [0]:
%sql

CREATE OR REPLACE VIEW ercot.stg_ercot_load_1h AS
SELECT
  hour_ts,
  AVG(ercot_mw) AS ercot_mw,
  AVG(coast_mw) AS coast_mw,
  AVG(east_mw)  AS east_mw,
  AVG(fwest_mw) AS fwest_mw,
  AVG(north_mw) AS north_mw,
  AVG(ncent_mw) AS ncent_mw,
  AVG(south_mw) AS south_mw,
  AVG(scent_mw) AS scent_mw,
  AVG(west_mw)  AS west_mw
FROM ercot.stg_ercot_load
GROUP BY hour_ts;


# Data Quality Checks

Sanity validation:
- No null timestamps
- Exactly one row per hour
- No negative MW values



In [0]:
%sql

-- Row count
SELECT COUNT(*) AS total_rows
FROM ercot.stg_ercot_load_1h;

-- Duplicate hour check
SELECT hour_ts, COUNT(*) AS cnt
FROM ercot.stg_ercot_load_1h
GROUP BY hour_ts
HAVING COUNT(*) > 1;

-- Negative load check
SELECT COUNT(*) AS negative_values
FROM ercot.stg_ercot_load_1h
WHERE ercot_mw < 0;


# Databricks notebook source


In [0]:
df = spark.table("ercot.stg_ercot_load_1h")

pdf = df.toPandas()

import base64

csv_text = pdf.to_csv(index=False)
b64 = base64.b64encode(csv_text.encode()).decode()

html = f"""
<a download="ercot_load_1h.csv"
   href="data:text/csv;base64,{b64}"
   style="font-size:16px; font-weight:600;">
   ⬇️ Download ercot_load_1h.csv
</a>
"""
displayHTML(html)


After export:

Download the generated CSV from:

https://community.cloud.databricks.com/files/ercot_export/ercot_load_1h/

Rename to:

    ercot_load_1h.csv

Place inside your local dbt repo:

    seeds/ercot_load_1h.csv
